### Support Vector Machine (SVM)


In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()

while not (project_root / "src").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Could not find project root containing 'src'")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: F:\CUFE\Data Science\Diabetes-Prediction


In [3]:
from sklearn.svm import SVC
import pandas as pd
from src.diabetes_prediction.transformation.transformation import DataTransformation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)


#### import Train Data
##### drop target ( Diabetes Column) 

In [4]:
train_df = pd.read_csv("../../data/split/train.csv")
test_df  = pd.read_csv("../../data/split/test.csv")
val_df = pd.read_csv("../../data/split/validation.csv")
x_val = val_df.drop("diabetes", axis = 1)
y_val = val_df["diabetes"]
x_train = train_df.drop("diabetes", axis=1)
y_train = train_df["diabetes"]
x_test = test_df.drop("diabetes", axis=1)
y_test = test_df["diabetes"]
x_train.head()


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level
0,Female,49.0,0,0,No Info,23.21,5.8,100
1,Female,59.0,0,0,former,32.60,6.5,140
2,Male,32.0,0,0,former,25.84,5.8,155
3,Female,70.0,0,0,ever,36.90,6.5,155
4,Male,55.0,0,0,never,24.77,3.5,200


### using DataTransformation class 

In [5]:
transformer = DataTransformation()
x_train = transformer.fit_transform(x_train)
transformer.save_preprocessor()
x_train.head()

,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,glucose_hba1c_interaction,age_hba1c_interaction,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag
0,0.0,0.0,0.0,0.0,0.0,0.612112,-0.637209,0.0,-0.677966,-0.459103,0.289448,-0.056612,-0.232854,-0.081827,0,0,0,0,0,0
1,0.0,0.0,1.0,0.0,0.0,0.737237,0.818605,0.5,0.0,0.411609,0.770087,0.658534,1.018393,0.557564,0,0,0,0,0,0
2,0.0,0.0,1.0,0.0,0.0,0.399399,-0.229457,0.0,0.254237,0.382586,-0.187803,-0.339001,0.014118,-0.070409,1,0,0,0,0,0
3,0.0,1.0,0.0,0.0,0.0,0.874875,1.485271,0.5,0.254237,0.668865,1.116167,1.25859,1.470922,1.050428,0,0,0,0,1,0
4,0.0,0.0,0.0,1.0,0.0,0.687187,-0.395349,-1.642857,1.016949,-0.14248,-0.154405,0.148131,-1.008759,1.078972,1,0,0,0,0,0


In [ ]:
transformer.load_preprocessor()

feature_names = transformer.preprocessor.get_feature_names_out()
print (feature_names)

x_val_processed = transformer.transform(x_val)
x_test_processed = transformer.transform(x_test)

x_val_processed.head()

['smoking_history_current' 'smoking_history_ever' 'smoking_history_former'
 'smoking_history_never' 'smoking_history_not current' 'age' 'bmi'
 'HbA1c_level' 'blood_glucose_level' 'glucose_hba1c_interaction'
 'age_hba1c_interaction' 'age_bmi_interaction' 'bmi_hba1c_interaction'
 'age_glucose_interaction' 'gender' 'hypertension' 'heart_disease'
 'high_hba1c_flag' 'senior_flag' 'cardio_risk_flag']


,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,glucose_hba1c_interaction,age_hba1c_interaction,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag
0,0.0,0.0,1.0,0.0,0.0,0.974975,1.035659,-0.071429,-0.847458,-0.635884,1.065828,1.321361,0.725342,0.321598,0,0,0,0,1,0
1,0.0,0.0,0.0,1.0,0.0,0.624625,-1.127132,-0.571429,0.338983,0.121372,0.123911,-0.179235,-0.789295,0.508088,0,0,0,0,0,0
2,0.0,0.0,0.0,1.0,0.0,0.662162,0.0,0.571429,-0.847458,-0.422164,0.60697,0.226013,0.506962,-0.106565,0,0,0,1,0,0
3,0.0,0.0,0.0,0.0,0.0,0.787287,0.0,-0.071429,0.322034,0.401847,0.651985,0.47455,0.108866,0.891912,1,1,0,0,1,1
4,0.0,0.0,0.0,1.0,0.0,0.787287,2.344186,0.142857,0.338983,0.543536,0.743466,1.341121,1.710381,0.903901,0,0,0,0,1,0


### Train the model

In [8]:

model = SVC(
    kernel='rbf',
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
    )
model.fit(x_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [9]:
x_test_processed = transformer.transform(x_test)
y_pred = model.predict(x_test_processed)

    


print("========== Model Evaluation ==========")

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\n========== Confusion Matrix ==========")

cm = confusion_matrix(y_test, y_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

print("\n========== Classification Report ==========")
print(classification_report(y_test, y_pred))

if hasattr(model, "predict_proba"):
    y_proba = model.predict_proba(x_test_processed)[:, 1]
    print("ROC-AUC :", roc_auc_score(y_test, y_proba))

========== Model Evaluation ==========
Accuracy : 0.8860161245482346
Precision: 0.4313944817300522
Recall   : 0.9095911949685535
F1 Score : 0.5852301466868993

========== Confusion Matrix ==========
[[11591  1525]
 [  115  1157]]

TN: 11591
FP: 1525
FN: 115
TP: 1157

========== Classification Report ==========
              precision    recall  f1-score   support

           0       0.99      0.88      0.93     13116
           1       0.43      0.91      0.59      1272

    accuracy                           0.89     14388
   macro avg       0.71      0.90      0.76     14388
weighted avg       0.94      0.89      0.90     14388

ROC-AUC : 0.9655975837759249
